In [10]:
import joblib
import pandas as pd

# Load saved objects
data = joblib.load("rul_model.pkl")

model = data["model"]
selector = data["selector"]

# Load dataset
test_df = pd.read_csv("../dataset/test_FD001.txt", sep=r"\s+", header=None)

# Column names
columns = ["engine_id","cycle","op1","op2","op3"] + [f"sensor{i}" for i in range(1,22)]
test_df.columns = columns

# ----- Feature Engineering -----

model_df = test_df.drop(columns=["engine_id"])

model_df["cycle_norm"] = model_df["cycle"] / model_df["cycle"].max()

for sensor in ["sensor7","sensor12","sensor15","sensor20"]:
    model_df[f"{sensor}_roll_mean"] = (
        model_df.groupby(test_df["engine_id"])[sensor]
        .rolling(5)
        .mean()
        .reset_index(level=0, drop=True)
    )

model_df.bfill(inplace=True)

# Apply same feature selector
X_test = selector.transform(model_df)

# Predict
predictions = model.predict(X_test)

# Create results
results = pd.DataFrame({
    "engine_id": test_df["engine_id"],
    "predicted_rul": predictions
})

# Keep last cycle prediction
final_results = results.groupby("engine_id").last().reset_index()

# Save
final_results.to_csv("predictions_fd001.csv", index=False)

print("Predictions saved successfully!")

Predictions saved successfully!
